<a href="https://colab.research.google.com/github/mashamm/goit-rdb-hw-03/blob/main/hw3_nyc_taxi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ДЗ 3. NYC Yellow Taxi: завантаження, типізація, аудит якості та EDA у PostgreSQL

**Студентка:** Samoilenko Mariia

**Датасет:** NYC TLC Trip Record Data — Yellow Taxi, січень 2024
([офіційна сторінка](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), файл
`yellow_tripdata_2024-01.parquet`, [словник даних](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf)).

**Вибірка:** 13 колонок, 100 000 рядків — відтворювана випадкова вибірка `sample(n=100_000, random_state=42)` з усього місяця
(перші 100 000 рядків файлу покривають лише 1–3 січня й не містять пропусків, тому не репрезентативні).

## Завдання 1. Setup і завантаження

In [ ]:
import glob
import os
import subprocess
import sys


def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


pip_install("psycopg2-binary", "sqlalchemy", "pandas", "pyarrow", "fasteners", "platformdirs", "psutil")

# pgserver 0.1.4 має wheel-файли лише до cp312; його розширення зібране в abi3,
# тому на новішому Python (Colab) встановлюємо cp312-wheel з тегом abi3.
if sys.version_info < (3, 13):
    pip_install("pgserver==0.1.4")
else:
    wheel_dir = "/tmp/pgserver_wheel"
    subprocess.check_call([
        sys.executable, "-m", "pip", "download", "-q", "pgserver==0.1.4",
        "--no-deps", "--only-binary=:all:", "--python-version", "3.12", "-d", wheel_dir,
    ])
    for whl in glob.glob(f"{wheel_dir}/pgserver-0.1.4-cp312-cp312-*.whl"):
        os.replace(whl, whl.replace("-cp312-cp312-", "-cp312-abi3-"))
    pip_install("--no-deps", *glob.glob(f"{wheel_dir}/pgserver-0.1.4-cp312-abi3-*.whl"))

In [ ]:
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

pg = pgserver.get_server("/tmp/hw3_pg", cleanup_mode="stop")
engine = create_engine(pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1), future=True)

with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar_one()
    print(version)


def q(sql_text):
    """Виконати SELECT і повернути результат як DataFrame."""
    return pd.read_sql(text(sql_text), engine)


def run(sql_text):
    """Виконати DDL/DML в одній транзакції."""
    with engine.begin() as conn:
        conn.execute(text(sql_text))

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


### Отримання даних з офіційного джерела

Завантажуємо один місячний Parquet-файл (не весь рік), залишаємо 13 потрібних колонок і формуємо контрольований CSV.
Часові колонки в Parquet — це «наївний» локальний час Нью-Йорка без часового поясу; у CSV вони записуються як
`YYYY-MM-DD HH:MM:SS`.

In [ ]:
from urllib.request import urlretrieve

DATA_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
DATA_DIR = "/content" if os.path.isdir("/content") else os.path.abspath("../data")
os.makedirs(DATA_DIR, exist_ok=True)

parquet_path = os.path.join(DATA_DIR, "hw3_taxi_2024_01.parquet")
if not os.path.exists(parquet_path):
    urlretrieve(DATA_URL, parquet_path)

columns = [
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "passenger_count", "trip_distance", "RatecodeID", "store_and_fwd_flag",
    "PULocationID", "DOLocationID", "payment_type",
    "fare_amount", "tip_amount", "total_amount",
]

full = pd.read_parquet(parquet_path, columns=columns)
df = full.sample(n=100_000, random_state=42).sort_values("tpep_pickup_datetime")

csv_path = os.path.join(DATA_DIR, "hw3_taxi_sample.csv")
df.to_csv(csv_path, index=False)

print("рядків у місячному файлі:", len(full))
print("вибірка:", df.shape)
df.head()

рядків у місячному файлі: 2964624
вибірка: (100000, 13)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,tip_amount,total_amount
256,2,2023-12-31 23:56:46,2024-01-01 00:12:06,2.0,2.38,1.0,N,236,142,1,15.6,1.00,21.60
4137,2,2023-12-31 23:58:35,2024-01-01 00:13:06,6.0,8.39,1.0,N,138,217,2,33.1,0.00,42.35
3782,2,2024-01-01 00:00:17,2024-01-01 00:07:46,1.0,2.35,1.0,N,161,75,1,12.1,3.42,20.52
5138,2,2024-01-01 00:01:39,2024-01-01 01:02:53,1.0,2.59,1.0,N,161,246,2,47.1,0.00,52.10
2936,2,2024-01-01 00:02:35,2024-01-01 00:10:55,1.0,1.36,1.0,N,90,79,1,10.0,3.00,18.00


### Staging-таблиця і `COPY FROM STDIN`

Усі сирі колонки мають тип `TEXT`, тому імпорт не впаде через порожні рядки, `1.0` замість цілого числа чи нестандартну дату.

In [ ]:
run("""
DROP TABLE IF EXISTS hw3_taxi_trips CASCADE;
DROP TABLE IF EXISTS hw3_taxi_staging CASCADE;

CREATE TABLE hw3_taxi_staging (
    vendor_id_raw          TEXT,
    pickup_raw             TEXT,
    dropoff_raw            TEXT,
    passenger_count_raw    TEXT,
    trip_distance_raw      TEXT,
    ratecode_id_raw        TEXT,
    store_and_fwd_flag_raw TEXT,
    pu_location_id_raw     TEXT,
    do_location_id_raw     TEXT,
    payment_type_raw       TEXT,
    fare_amount_raw        TEXT,
    tip_amount_raw         TEXT,
    total_amount_raw       TEXT
);
""")

raw_conn = engine.raw_connection()
try:
    with raw_conn.cursor() as cur:
        with open(csv_path, "r", encoding="utf-8", newline="") as file:
            cur.copy_expert(
                """
                COPY hw3_taxi_staging (
                    vendor_id_raw, pickup_raw, dropoff_raw, passenger_count_raw,
                    trip_distance_raw, ratecode_id_raw, store_and_fwd_flag_raw,
                    pu_location_id_raw, do_location_id_raw, payment_type_raw,
                    fare_amount_raw, tip_amount_raw, total_amount_raw
                )
                FROM STDIN
                WITH (FORMAT csv, HEADER true, DELIMITER ',')
                """,
                file,
            )
    raw_conn.commit()
finally:
    raw_conn.close()

q("SELECT COUNT(*) AS loaded_rows FROM hw3_taxi_staging")

,loaded_rows
0,100000


## Завдання 2. Staging → типізована таблиця

| Колонка | Семантика | Джерельний формат | Тип | Правила очищення |
|---|---|---|---|---|
| `trip_id` | технічний surrogate key | — | `BIGINT IDENTITY PK` | генерується БД; природного ключа поїздки у файлі немає |
| `vendor_id` | постачальник TPEP-системи (1 = Creative Mobile, 2 = VeriFone) | ціле число як текст | `SMALLINT NOT NULL` | `TRIM` → `NULLIF('')` → `::SMALLINT` |
| `pickup_at`, `dropoff_at` | моменти посадки й висадки | `YYYY-MM-DD HH:MM:SS`, локальний час Нью-Йорка без поясу | `TIMESTAMPTZ NOT NULL` | `::TIMESTAMP AT TIME ZONE 'America/New_York'` — момент прив'язується до поясу NYC (з урахуванням DST) |
| `pickup_date` | календарна дата поїздки в NYC | похідна від `pickup_raw` | `DATE NOT NULL` | `::TIMESTAMP::DATE` — локальна дата без часу для денних агрегатів |
| `passenger_count` | кількість пасажирів (вводить водій) | дробове число `1.0` або порожньо | `SMALLINT`, `CHECK >= 0` | `::NUMERIC::SMALLINT`; порожньо → `NULL` (не підставляємо 1) |
| `trip_distance` | відстань за таксометром, милі | десяткове | `NUMERIC(10,2) NOT NULL`, `CHECK >= 0` | рядки з від'ємною відстанню відкидаються |
| `rate_code_id` | тариф (1 standard, 2 JFK, 3 Newark, 4 Nassau/Westchester, 5 negotiated, 6 group, 99 unknown) | дробове `1.0` або порожньо | `SMALLINT`, `CHECK IN (1..6, 99)` | `::NUMERIC::SMALLINT`; `99` зберігаємо як явний маркер «невідомо» |
| `store_and_fwd` | чи зберігався запис у пам'яті авто без зв'язку з сервером | `Y` / `N` / порожньо | `BOOLEAN` | `Y → TRUE`, `N → FALSE`, інше → `NULL`; справжня двійкова ознака |
| `pu_location_id`, `do_location_id` | TLC Taxi Zone посадки / висадки | ціле | `INTEGER NOT NULL`, `CHECK BETWEEN 1 AND 265` | `::INTEGER`; 264/265 — «Unknown / Outside NYC» за довідником зон |
| `payment_type` | спосіб оплати (0 flex/невідомо, 1 card, 2 cash, 3 no charge, 4 dispute, 5 unknown, 6 voided) | ціле | `SMALLINT NOT NULL`, `CHECK BETWEEN 0 AND 6` | `::SMALLINT` |
| `fare_amount`, `tip_amount`, `total_amount` | тариф за таксометром, чайові, загальна сума, USD | десяткове | `NUMERIC(10,2)` | грошові суми з точністю до центів; від'ємні значення (коригування/повернення) **не** відкидаємо, а аналізуємо в аудиті |

**Правило фільтрації** під час `INSERT INTO ... SELECT`: відкидаються рядки без часу посадки/висадки, з `dropoff_at < pickup_at`,
з від'ємною відстанню, а також рядки, що порушують доменні `CHECK` (невідомий `rate_code_id`, зона поза 1–265, `payment_type` поза 0–6).

In [ ]:
run("""
DROP TABLE IF EXISTS hw3_taxi_trips CASCADE;

CREATE TABLE hw3_taxi_trips (
    trip_id          BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    vendor_id        SMALLINT       NOT NULL,
    pickup_at        TIMESTAMPTZ    NOT NULL,
    dropoff_at       TIMESTAMPTZ    NOT NULL,
    pickup_date      DATE           NOT NULL,
    passenger_count  SMALLINT       CHECK (passenger_count IS NULL OR passenger_count >= 0),
    trip_distance    NUMERIC(10, 2) NOT NULL CHECK (trip_distance >= 0),
    rate_code_id     SMALLINT       CHECK (rate_code_id IS NULL OR rate_code_id IN (1, 2, 3, 4, 5, 6, 99)),
    store_and_fwd    BOOLEAN,
    pu_location_id   INTEGER        NOT NULL CHECK (pu_location_id BETWEEN 1 AND 265),
    do_location_id   INTEGER        NOT NULL CHECK (do_location_id BETWEEN 1 AND 265),
    payment_type     SMALLINT       NOT NULL CHECK (payment_type BETWEEN 0 AND 6),
    fare_amount      NUMERIC(10, 2),
    tip_amount       NUMERIC(10, 2),
    total_amount     NUMERIC(10, 2),

    CHECK (dropoff_at >= pickup_at)
);

WITH parsed AS (
    SELECT
        NULLIF(TRIM(vendor_id_raw), '')::SMALLINT                               AS vendor_id,
        NULLIF(TRIM(pickup_raw), '')::TIMESTAMP AT TIME ZONE 'America/New_York'  AS pickup_at,
        NULLIF(TRIM(dropoff_raw), '')::TIMESTAMP AT TIME ZONE 'America/New_York' AS dropoff_at,
        NULLIF(TRIM(pickup_raw), '')::TIMESTAMP::DATE                           AS pickup_date,
        NULLIF(TRIM(passenger_count_raw), '')::NUMERIC::SMALLINT                AS passenger_count,
        NULLIF(TRIM(trip_distance_raw), '')::NUMERIC(10, 2)                     AS trip_distance,
        NULLIF(TRIM(ratecode_id_raw), '')::NUMERIC::SMALLINT                    AS rate_code_id,
        CASE UPPER(TRIM(store_and_fwd_flag_raw))
            WHEN 'Y' THEN TRUE
            WHEN 'N' THEN FALSE
        END                                                                     AS store_and_fwd,
        NULLIF(TRIM(pu_location_id_raw), '')::INTEGER                           AS pu_location_id,
        NULLIF(TRIM(do_location_id_raw), '')::INTEGER                           AS do_location_id,
        NULLIF(TRIM(payment_type_raw), '')::SMALLINT                            AS payment_type,
        NULLIF(TRIM(fare_amount_raw), '')::NUMERIC(10, 2)                       AS fare_amount,
        NULLIF(TRIM(tip_amount_raw), '')::NUMERIC(10, 2)                        AS tip_amount,
        NULLIF(TRIM(total_amount_raw), '')::NUMERIC(10, 2)                      AS total_amount
    FROM hw3_taxi_staging
)
INSERT INTO hw3_taxi_trips (
    vendor_id, pickup_at, dropoff_at, pickup_date, passenger_count, trip_distance,
    rate_code_id, store_and_fwd, pu_location_id, do_location_id, payment_type,
    fare_amount, tip_amount, total_amount
)
SELECT
    vendor_id, pickup_at, dropoff_at, pickup_date, passenger_count, trip_distance,
    rate_code_id, store_and_fwd, pu_location_id, do_location_id, payment_type,
    fare_amount, tip_amount, total_amount
FROM parsed
WHERE vendor_id IS NOT NULL
  AND pickup_at IS NOT NULL
  AND dropoff_at IS NOT NULL
  AND dropoff_at >= pickup_at
  AND trip_distance >= 0
  AND (passenger_count IS NULL OR passenger_count >= 0)
  AND (rate_code_id IS NULL OR rate_code_id IN (1, 2, 3, 4, 5, 6, 99))
  AND pu_location_id BETWEEN 1 AND 265
  AND do_location_id BETWEEN 1 AND 265
  AND payment_type BETWEEN 0 AND 6;
""")

q("SELECT * FROM hw3_taxi_trips ORDER BY trip_id LIMIT 5")

,trip_id,vendor_id,pickup_at,dropoff_at,pickup_date,passenger_count,trip_distance,rate_code_id,store_and_fwd,pu_location_id,do_location_id,payment_type,fare_amount,tip_amount,total_amount
0,1,2,2024-01-01 04:56:46+00:00,2024-01-01 05:12:06+00:00,2023-12-31,2,2.38,1,False,236,142,1,15.6,1.00,21.60
1,2,2,2024-01-01 04:58:35+00:00,2024-01-01 05:13:06+00:00,2023-12-31,6,8.39,1,False,138,217,2,33.1,0.00,42.35
2,3,2,2024-01-01 05:00:17+00:00,2024-01-01 05:07:46+00:00,2024-01-01,1,2.35,1,False,161,75,1,12.1,3.42,20.52
3,4,2,2024-01-01 05:01:39+00:00,2024-01-01 06:02:53+00:00,2024-01-01,1,2.59,1,False,161,246,2,47.1,0.00,52.10
4,5,2,2024-01-01 05:02:35+00:00,2024-01-01 05:10:55+00:00,2024-01-01,1,1.36,1,False,90,79,1,10.0,3.00,18.00


## Завдання 3. Data-quality audit

### 3.1. Контроль кількості рядків

In [ ]:
q("""
SELECT 'staging' AS layer, COUNT(*) AS n_rows FROM hw3_taxi_staging
UNION ALL
SELECT 'clean' AS layer, COUNT(*) AS n_rows FROM hw3_taxi_trips
""")

,layer,n_rows
0,staging,100000
1,clean,99999


Причини відсіювання (рахуються на сирому шарі за тими самими правилами):

In [ ]:
q("""
SELECT
    COUNT(*) FILTER (WHERE dropoff_raw::TIMESTAMP < pickup_raw::TIMESTAMP)       AS dropoff_before_pickup,
    COUNT(*) FILTER (WHERE trip_distance_raw::NUMERIC < 0)                        AS negative_distance,
    COUNT(*) FILTER (WHERE pu_location_id_raw::INTEGER NOT BETWEEN 1 AND 265
                        OR do_location_id_raw::INTEGER NOT BETWEEN 1 AND 265)     AS bad_location,
    COUNT(*) FILTER (WHERE payment_type_raw::INTEGER NOT BETWEEN 0 AND 6)         AS bad_payment_type,
    COUNT(*) FILTER (WHERE NULLIF(TRIM(ratecode_id_raw), '')::NUMERIC
                           NOT IN (1, 2, 3, 4, 5, 6, 99))                         AS bad_rate_code
FROM hw3_taxi_staging
""")

,dropoff_before_pickup,negative_distance,bad_location,bad_payment_type,bad_rate_code
0,1,0,0,0,0


**Висновок.** У staging завантажено 100 000 рядків, у clean — 99 999. Відсіяно рівно один рядок за правилом `dropoff_at >= pickup_at` (висадка раніше посадки — фізично неможливий запис). Інші правила (від'ємна відстань, зона поза 1–265, `payment_type` поза 0–6, невідомий тариф) у цій вибірці не спрацювали.

### 3.2. Перевірка дублікатів

Гарантованого природного ключа поїздки у файлі немає. Як business key беремо комбінацію
`(vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id, total_amount)`: дві різні поїздки одного
постачальника з однаковими секундами посадки й висадки, зонами та сумою практично неможливі. Тож перевірка виявляє
**точні або потенційні** дублікати, а не гарантовані.

In [ ]:
q("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id, total_amount))
        AS distinct_business_keys,
    COUNT(*) - COUNT(DISTINCT (vendor_id, pickup_at, dropoff_at, pu_location_id, do_location_id, total_amount))
        AS duplicate_rows
FROM hw3_taxi_trips
""")

,total_rows,distinct_business_keys,duplicate_rows
0,99999,99999,0


**Висновок.** Усі 99 999 business keys унікальні, точних дублікатів немає. Це не гарантує відсутності логічних дублів (наприклад, повторна передача запису з іншою сумою після коригування), бо справжнього ідентифікатора поїздки файл не містить.

### 3.3. Null-rate і спеціальні маркери

Null-rate у clean-шарі для семи колонок:

In [ ]:
q("""
SELECT
    ROUND(COUNT(*) FILTER (WHERE passenger_count IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_passenger_count,
    ROUND(COUNT(*) FILTER (WHERE rate_code_id    IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_rate_code_id,
    ROUND(COUNT(*) FILTER (WHERE store_and_fwd   IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_store_and_fwd,
    ROUND(COUNT(*) FILTER (WHERE trip_distance   IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_trip_distance,
    ROUND(COUNT(*) FILTER (WHERE fare_amount     IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_fare_amount,
    ROUND(COUNT(*) FILTER (WHERE tip_amount      IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_tip_amount,
    ROUND(COUNT(*) FILTER (WHERE total_amount    IS NULL) * 100.0 / NULLIF(COUNT(*), 0), 2) AS pct_null_total_amount,
    COUNT(*) AS total_rows
FROM hw3_taxi_trips
""")

,pct_null_passenger_count,pct_null_rate_code_id,pct_null_store_and_fwd,pct_null_trip_distance,pct_null_fare_amount,pct_null_tip_amount,pct_null_total_amount,total_rows
0,4.73,4.73,4.73,0.0,0.0,0.0,0.0,99999


Спеціальні маркери в сирому шарі: порожні значення (COPY перетворює порожнє поле CSV на NULL, тому рахуємо і NULL, і `''`), коди «невідомо» (`payment_type = 0/5`, `RatecodeID = 99`), зони 264/265 (Unknown/Outside NYC):

In [ ]:
q("""
SELECT
    COUNT(*) FILTER (WHERE COALESCE(TRIM(passenger_count_raw), '') = '')    AS empty_passenger_count,
    COUNT(*) FILTER (WHERE COALESCE(TRIM(ratecode_id_raw), '') = '')        AS empty_ratecode,
    COUNT(*) FILTER (WHERE COALESCE(TRIM(store_and_fwd_flag_raw), '') = '') AS empty_store_and_fwd,
    COUNT(*) FILTER (WHERE payment_type_raw IN ('0', '5'))     AS payment_unknown_0_or_5,
    COUNT(*) FILTER (WHERE ratecode_id_raw = '99.0')           AS ratecode_99_unknown,
    COUNT(*) FILTER (WHERE pu_location_id_raw IN ('264', '265')
                        OR do_location_id_raw IN ('264', '265')) AS unknown_zone_264_265,
    COUNT(*) FILTER (WHERE passenger_count_raw = '0.0')        AS zero_passengers,
    COUNT(*)                                                   AS total_rows
FROM hw3_taxi_staging
""")

,empty_passenger_count,empty_ratecode,empty_store_and_fwd,payment_unknown_0_or_5,ratecode_99_unknown,unknown_zone_264_265,zero_passengers,total_rows
0,4727,4727,4727,4727,962,1020,1085,100000


**Висновок.** Грошові колонки й відстань не мають пропусків (0 %). `passenger_count`, `rate_code_id` і `store_and_fwd` мають однаковий null-rate 4,73 %: це ті самі рядки з `payment_type = 0` (4 727 записів, flex fare), для яких постачальник не передає ці поля. Окрім NULL, у даних є «приховані» пропуски: 962 рядки з `RatecodeID = 99` (невідомий тариф), 1 020 поїздок із зоною 264/265 (Unknown / Outside NYC) і 1 085 поїздок з `passenger_count = 0`, що для реальної поїздки неможливо. Нульовий null-rate у грошових полях не означає їхньої якості — нижче видно від'ємні суми.

## Завдання 4. Розподіли та потенційні аномалії

### 4.1. Розподіл способів оплати

In [ ]:
q("""
SELECT
    payment_type,
    CASE payment_type
        WHEN 0 THEN 'flex fare / unknown'
        WHEN 1 THEN 'credit card'
        WHEN 2 THEN 'cash'
        WHEN 3 THEN 'no charge'
        WHEN 4 THEN 'dispute'
        WHEN 5 THEN 'unknown'
        WHEN 6 THEN 'voided trip'
    END AS payment_label,
    COUNT(*) AS n_rows,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_rows
FROM hw3_taxi_trips
GROUP BY payment_type
ORDER BY n_rows DESC, payment_type
LIMIT 10
""")

,payment_type,payment_label,n_rows,pct_rows
0,1,credit card,78131,78.13
1,2,cash,14877,14.88
2,0,flex fare / unknown,4727,4.73
3,4,dispute,1594,1.59
4,3,no charge,670,0.67


**Висновок.** Розподіл сильно незбалансований: 78,1 % поїздок оплачено карткою, 14,9 % — готівкою. Код 0 (flex fare / невідомо, 4,7 %) — це не реальний спосіб оплати, а маркер відсутньої інформації. Рідкісні категорії dispute (1,6 %) і no charge (0,7 %) важливі для якості даних, бо саме в них зосереджені від'ємні суми. Для моделі чайових картка домінує, а готівкові поїздки слід обробляти окремо (див. Q7).

### 4.2. Базова статистика: `trip_distance` і `total_amount`

In [ ]:
q("""
SELECT 'trip_distance' AS metric,
       COUNT(trip_distance) AS n_non_null, MIN(trip_distance) AS min_value, MAX(trip_distance) AS max_value,
       ROUND(AVG(trip_distance), 2) AS avg_value, ROUND(STDDEV_SAMP(trip_distance), 2) AS sd_value
FROM hw3_taxi_trips
UNION ALL
SELECT 'total_amount',
       COUNT(total_amount), MIN(total_amount), MAX(total_amount),
       ROUND(AVG(total_amount), 2), ROUND(STDDEV_SAMP(total_amount), 2)
FROM hw3_taxi_trips
""")

,metric,n_non_null,min_value,max_value,avg_value,sd_value
0,trip_distance,99999,0.00,99.00,3.21,4.33
1,total_amount,99999,-188.71,623.71,26.68,22.67


**Висновок.** Середня відстань 3,21 милі при стандартному відхиленні 4,33 і максимумі 99 милях: розподіл сильно скошений вправо (багато коротких міських поїздок і довгий хвіст аеропортних). `total_amount` коливається від −188,71 до 623,71 USD при середньому 26,68; від'ємний мінімум уже вказує на коригування. Через скошеність правило 3σ тут малоінформативне, тому нижче використано доменні перевірки.

### 4.3. Domain outlier-check

Правила, що суперечать предметній області: від'ємні суми, дата посадки поза січнем 2024, поїздка довша за 12 годин,
відстань понад 100 миль, ненульова оплата при нульовій відстані й тривалості менше хвилини.

In [ ]:
q("""
SELECT
    COUNT(*) FILTER (WHERE fare_amount < 0 OR tip_amount < 0 OR total_amount < 0)             AS negative_amounts,
    COUNT(*) FILTER (WHERE pickup_date NOT BETWEEN DATE '2024-01-01' AND DATE '2024-01-31')    AS pickup_outside_month,
    COUNT(*) FILTER (WHERE dropoff_at - pickup_at > INTERVAL '12 hours')                       AS longer_than_12h,
    COUNT(*) FILTER (WHERE trip_distance > 100)                                                AS distance_over_100mi,
    COUNT(*) FILTER (WHERE trip_distance = 0 AND dropoff_at - pickup_at < INTERVAL '1 minute'
                       AND total_amount > 0)                                                   AS zero_trip_but_charged
FROM hw3_taxi_trips
""")

,negative_amounts,pickup_outside_month,longer_than_12h,distance_over_100mi,zero_trip_but_charged
0,1308,2,38,0,712


In [ ]:
q("""
SELECT trip_id, pickup_at, dropoff_at, trip_distance, payment_type, fare_amount, tip_amount, total_amount
FROM hw3_taxi_trips
WHERE fare_amount < 0
   OR total_amount < 0
   OR pickup_date NOT BETWEEN DATE '2024-01-01' AND DATE '2024-01-31'
   OR trip_distance > 100
ORDER BY pickup_at
LIMIT 20
""")

,trip_id,pickup_at,dropoff_at,trip_distance,payment_type,fare_amount,tip_amount,total_amount
0,1,2024-01-01 04:56:46+00:00,2024-01-01 05:12:06+00:00,2.38,1,15.60,1.0,21.60
1,2,2024-01-01 04:58:35+00:00,2024-01-01 05:13:06+00:00,8.39,2,33.10,0.0,42.35
2,49,2024-01-01 05:24:02+00:00,2024-01-01 06:10:32+00:00,8.74,4,-47.80,0.0,-52.80
3,64,2024-01-01 05:28:42+00:00,2024-01-01 05:36:20+00:00,0.65,4,-7.90,0.0,-12.90
4,104,2024-01-01 05:36:26+00:00,2024-01-01 05:39:25+00:00,0.49,4,-5.10,0.0,-10.10
5,119,2024-01-01 05:40:25+00:00,2024-01-01 05:46:13+00:00,0.50,2,-70.00,0.0,-74.00
6,190,2024-01-01 05:57:17+00:00,2024-01-01 06:10:43+00:00,4.92,0,-0.42,0.0,3.58
7,289,2024-01-01 06:20:14+00:00,2024-01-01 06:24:35+00:00,0.59,3,-5.80,0.0,-10.80
8,305,2024-01-01 06:23:43+00:00,2024-01-01 06:33:11+00:00,0.74,4,-10.00,0.0,-15.00
9,384,2024-01-01 06:44:20+00:00,2024-01-01 06:47:42+00:00,0.50,2,-5.80,0.0,-10.80


**Висновок і дії.**

- **1 308 поїздок з від'ємними сумами** — переважно `payment_type` 4 (dispute) і 3 (no charge), тобто сторно/повернення. Дія: **залишити в сховищі з прапорцем** і **виключити з training data** для моделей тарифу/чайових; уточнити в джерела, чи для кожного сторно є пара-оригінал.
- **2 поїздки з датою посадки 31.12.2023** у файлі за січень — дійсні поїздки через північ. Дія: **залишити як валідний рідкісний випадок**; для помісячних агрегатів фільтрувати за `pickup_date`.
- **38 поїздок довших за 12 годин** — ймовірно, таксометр не вимкнули. Дія: **додати прапорець** і **передати на перевірку джерела**; тривалість таких поїздок не використовувати як ознаку.
- **712 поїздок з нульовою відстанню, тривалістю менше хвилини й ненульовою оплатою** — скасовані посадки або мінімальний тариф. Дія: **додати прапорець**, не видаляти автоматично.
- Відстаней понад 100 миль не знайдено (максимум 99): великі значення самі по собі не є помилкою, тому їх не видаляємо.

## Завдання 5. Десять DQL/EDA-запитів

### Q1

**Business-question:** У які години доби (за часом Нью-Йорка) починається найбільше поїздок?

In [ ]:
q("""
SELECT
    EXTRACT(HOUR FROM pickup_at AT TIME ZONE 'America/New_York')::INT AS pickup_hour,
    COUNT(*) AS n_trips
FROM hw3_taxi_trips
GROUP BY pickup_hour
ORDER BY n_trips DESC, pickup_hour
LIMIT 5
""")

,pickup_hour,n_trips
0,18,7082
1,17,6946
2,16,6487
3,15,6476
4,19,6290


**Interpretation:** Пік попиту припадає на вечірні години: найбільше поїздок починається о 18:00 (7 082), далі 17:00 (6 946) і 16:00 (6 487); у топ-5 також 15:00 і 19:00. Отже, попит концентрується в денний і вечірній «час пік». Година посадки — корисна ознака для прогнозу попиту й тривалості поїздки.

### Q2

**Business-question:** Яка частка поїздок кожного постачальника має невідому кількість пасажирів (NULL passenger_count)?

In [ ]:
q("""
SELECT
    vendor_id,
    COUNT(*) AS n_trips,
    COUNT(*) FILTER (WHERE passenger_count IS NULL) AS n_null_passengers,
    ROUND(COUNT(*) FILTER (WHERE passenger_count IS NULL) * 100.0 / COUNT(*), 2) AS pct_null_passengers
FROM hw3_taxi_trips
GROUP BY vendor_id
ORDER BY vendor_id
""")

,vendor_id,n_trips,n_null_passengers,pct_null_passengers
0,1,24784,1623,6.55
1,2,75209,3098,4.12
2,6,6,6,100.00


**Interpretation:** У Vendor 1 невідома кількість пасажирів трапляється в 6,55 % поїздок, у Vendor 2 — в 4,12 %. Несподівано з'явився vendor_id = 6 (6 поїздок, усі з NULL): у словнику TLC 2024 року цей код додано для нового постачальника, тому його треба додати до довідника, а не вважати помилкою. Пропуски залежать від постачальника, отже це системна особливість pipeline, а не випадковий шум.

### Q3

**Business-question:** Які поїздки мають від'ємні суми або нульову відстань, хоча це не безкоштовна й не спірна поїздка?

In [ ]:
q("""
SELECT
    trip_id, pickup_at, trip_distance, payment_type, fare_amount, total_amount
FROM hw3_taxi_trips
WHERE (fare_amount < 0 OR total_amount < 0)
   OR (trip_distance = 0 AND NOT (payment_type IN (3, 4)) AND total_amount > 0)
ORDER BY total_amount
LIMIT 10
""")

,trip_id,pickup_at,trip_distance,payment_type,fare_amount,total_amount
0,68053,2024-01-22 21:37:55+00:00,31.26,4,-154.2,-188.71
1,13777,2024-01-05 22:34:52+00:00,21.03,4,-170.0,-187.50
2,73156,2024-01-24 14:35:12+00:00,0.00,4,-150.0,-151.00
3,46102,2024-01-16 07:08:14+00:00,27.29,4,-130.4,-141.59
4,6548,2024-01-03 19:57:51+00:00,20.52,2,-99.3,-129.80
5,18302,2024-01-07 04:16:46+00:00,29.02,4,-110.1,-116.85
6,66058,2024-01-22 01:49:17+00:00,23.79,4,-103.1,-114.29
7,99667,2024-02-01 02:49:10+00:00,27.79,4,-102.4,-106.65
8,20672,2024-01-08 03:53:51+00:00,25.17,4,-91.2,-100.45
9,5400,2024-01-03 12:13:26+00:00,24.67,2,-96.1,-100.10


**Interpretation:** Найбільші від'ємні суми (до −188,71 USD) мають поїздки з `payment_type = 4` (dispute) і кілька готівкових поїздок з `payment_type = 2`. Це повернення коштів за довгі (20–31 миля) поїздки. Запис на 0 миль із −151 USD — сторно без поїздки. Такі рядки не можна використовувати як звичайні тарифи: вони потребують прапорця або виключення з training data.

### Q4

**Business-question:** Чим відрізняються середній тариф і чайові для поїздок з аеропортів JFK (132) і LaGuardia (138) від решти?

In [ ]:
q("""
SELECT
    CASE WHEN pu_location_id IN (132, 138) THEN 'airport pickup' ELSE 'other' END AS pickup_group,
    COUNT(*) AS n_trips,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(fare_amount), 2) AS avg_fare,
    ROUND(AVG(tip_amount), 2) AS avg_tip
FROM hw3_taxi_trips
GROUP BY pickup_group
ORDER BY n_trips DESC
""")

,pickup_group,n_trips,avg_distance,avg_fare,avg_tip
0,other,92240,2.37,15.21,2.86
1,airport pickup,7759,13.16,52.37,8.83


**Interpretation:** Поїздки з аеропортів JFK і LaGuardia становлять 7 759 із 99 999 (≈ 7,8 %), але вони в середньому в 5,5 раза довші (13,16 проти 2,37 милі) і втричі дорожчі (тариф 52,37 проти 15,21 USD). Середні чайові теж утричі вищі (8,83 проти 2,86 USD). Зона посадки — сильна ознака для прогнозу вартості, а аеропортні поїздки варто враховувати як окремий сегмент.

### Q5

**Business-question:** Яку частку становлять довгі поїздки на 10–30 миль і скільки вони в середньому коштують?

In [ ]:
q("""
SELECT
    COUNT(*) FILTER (WHERE trip_distance BETWEEN 10 AND 30) AS n_long_trips,
    ROUND(COUNT(*) FILTER (WHERE trip_distance BETWEEN 10 AND 30) * 100.0 / COUNT(*), 2) AS pct_long_trips,
    ROUND(AVG(total_amount) FILTER (WHERE trip_distance BETWEEN 10 AND 30), 2) AS avg_total_long,
    ROUND(AVG(total_amount) FILTER (WHERE trip_distance < 10), 2) AS avg_total_short
FROM hw3_taxi_trips
""")

,n_long_trips,pct_long_trips,avg_total_long,avg_total_short
0,7629,7.63,81.62,21.94


**Interpretation:** Поїздки на 10–30 миль становлять 7,63 % (7 629), але коштують у середньому 81,62 USD проти 21,94 USD для поїздок до 10 миль. Тобто невелика частка довгих поїздок формує помітну частину виручки. Для регресії вартості це аргумент на користь нелінійних моделей або логарифмування цільової змінної.

### Q6

**Business-question:** Скільки поїздок у файлі за січень 2024 насправді датовані іншим місяцем?

In [ ]:
q("""
SELECT
    pickup_date,
    COUNT(*) AS n_trips
FROM hw3_taxi_trips
WHERE pickup_date::TEXT NOT LIKE '2024-01-%'
GROUP BY pickup_date
ORDER BY pickup_date
""")

,pickup_date,n_trips
0,2023-12-31,2


**Interpretation:** Лише 2 поїздки в січневому файлі мають локальну дату посадки поза січнем — 31.12.2023 (посадка незадовго до півночі, висадка вже в січні). Якість датування у вибірці висока, але для помісячних агрегатів треба фільтрувати за `pickup_date`, а не покладатися на назву файлу.

### Q7

**Business-question:** Який середній відсоток чайових від тарифу при оплаті карткою та готівкою?

In [ ]:
q("""
SELECT
    payment_type,
    COUNT(*) AS n_trips,
    ROUND(AVG(tip_amount / NULLIF(fare_amount, 0)) * 100, 2) AS avg_tip_pct,
    ROUND(AVG(COALESCE(passenger_count, 1)), 2) AS avg_passengers_null_as_1
FROM hw3_taxi_trips
WHERE payment_type IN (1, 2)
  AND fare_amount > 0
GROUP BY payment_type
ORDER BY payment_type
""")

,payment_type,n_trips,avg_tip_pct,avg_passengers_null_as_1
0,1,78125,25.8,1.33
1,2,14592,0.0,1.39


**Interpretation:** Для оплати карткою середні чайові становлять 25,8 % тарифу, для готівки — 0 %. Це не означає, що за готівку не дають чайових: TLC фіксує лише чайові, сплачені карткою. Тому `tip_amount` для готівкових поїздок — систематично занижена ціль, і модель чайових слід навчати лише на карткових поїздках. Середня кількість пасажирів (NULL = 1) близька в обох групах: 1,33 і 1,39.

### Q8

**Business-question:** Яка частка поїздок починається й закінчується в тій самій зоні і яка в них середня відстань?

In [ ]:
q("""
SELECT
    pu_location_id IS NOT DISTINCT FROM do_location_id AS same_zone,
    COUNT(*) AS n_trips,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_trips,
    ROUND(AVG(trip_distance), 2) AS avg_distance
FROM hw3_taxi_trips
GROUP BY same_zone
ORDER BY same_zone
""")

,same_zone,n_trips,pct_trips,avg_distance
0,False,94529,94.53,3.35
1,True,5470,5.47,0.71


**Interpretation:** 5,47 % поїздок (5 470) починаються й закінчуються в тій самій зоні, і їхня середня відстань лише 0,71 милі проти 3,35 для інших. Це короткі внутрішньорайонні поїздки. Порівняння через `IS NOT DISTINCT FROM` безпечне для NULL, хоча в clean-шарі зони обов'язкові.

### Q9

**Business-question:** Які комбінації тарифу й способу оплати трапляються в поїздках із невідомою кількістю пасажирів?

In [ ]:
q("""
SELECT DISTINCT
    rate_code_id,
    payment_type,
    store_and_fwd
FROM hw3_taxi_trips
WHERE passenger_count IS NULL
ORDER BY rate_code_id NULLS FIRST, payment_type
""")

,rate_code_id,payment_type,store_and_fwd
0,None,0,None


**Interpretation:** Усі поїздки з невідомою кількістю пасажирів мають одну й ту саму комбінацію: `payment_type = 0`, а `rate_code_id` і `store_and_fwd` теж порожні. Отже, пропуски не випадкові: це окремий тип записів (flex fare), для якого постачальник не передає блок атрибутів. Такі рядки краще позначати окремою категорією, ніж заповнювати пропуски середніми значеннями.

### Q10

**Business-question:** Які маршрути «зона посадки → зона висадки» найпопулярніші, якщо виключити невідомі зони 264/265?

In [ ]:
q("""
SELECT
    pu_location_id,
    do_location_id,
    COUNT(*) AS n_trips,
    ROUND(AVG(total_amount), 2) AS avg_total
FROM hw3_taxi_trips
WHERE NOT (pu_location_id IN (264, 265) OR do_location_id IN (264, 265))
  AND pu_location_id IS DISTINCT FROM do_location_id
GROUP BY pu_location_id, do_location_id
ORDER BY n_trips DESC, pu_location_id, do_location_id
LIMIT 10
""")

,pu_location_id,do_location_id,n_trips,avg_total
0,237,236,703,15.86
1,236,237,639,15.75
2,161,237,325,16.54
3,239,238,308,13.68
4,237,161,294,16.04
5,239,142,292,14.31
6,142,239,282,15.13
7,161,236,279,21.23
8,141,236,274,15.64
9,236,161,265,20.56


**Interpretation:** Найпопулярніші маршрути — між сусідніми зонами Upper East Side 237 → 236 (703 поїздки) і 236 → 237 (639), далі Midtown 161 → 237 (325). Середня вартість на цих маршрутах — 14–21 USD, тобто це короткі поїздки Мангеттеном. Пара «зона посадки → зона висадки» концентрує попит і є корисною ознакою для прогнозу тривалості й вартості.

### Покриття обов'язкових конструкцій

| Конструкція | Запити |
|---|---|
| явний перелік колонок у `SELECT` | Q3, Q6, Q9, Q10 (і всі інші) |
| `WHERE` з `AND` / `OR` / `NOT` і дужками | Q3, Q10 |
| `IN` / `BETWEEN` | Q3, Q4, Q7, Q10 / Q5 |
| `LIKE` | Q6 |
| `IS NULL` / `IS NOT NULL` | Q2, Q9 |
| `COALESCE` / `NULLIF` | Q7 |
| `IS DISTINCT FROM` / `IS NOT DISTINCT FROM` | Q10 / Q8 |
| `DISTINCT` | Q9 |
| `ORDER BY` + `LIMIT` | Q1, Q3, Q10 |
| `GROUP BY` (≥ 2) | Q1, Q2, Q4, Q6, Q7, Q8, Q10 |

## Завдання 6. Reflection

**Проблеми якості.** Грошові поля не мають NULL, але 1,3 % поїздок мають від'ємні суми (сторно в категоріях dispute / no charge). Пропуски `passenger_count`, `RatecodeID` і `store_and_fwd_flag` (4,7 %) не випадкові: це ті самі рядки з `payment_type = 0`. Є й неявні маркери «невідомо»: `RatecodeID = 99`, зони 264/265, `passenger_count = 0`, а також новий `vendor_id = 6`, якого немає в старих довідниках. Дублікатів за business key немає, але поїздки понад 12 годин і нульові поїздки з оплатою потребують прапорців.

**Features.** Годину й день тижня посадки, зони посадки й висадки, `rate_code_id`, `vendor_id` та `trip_distance` я використала б як ознаки (відстань — лише якщо прогноз робиться для запланованого маршруту). Не використала б `trip_id` (технічний ключ), `store_and_fwd` (технічний прапорець зв'язку) і `tip_amount` для готівкових поїздок (систематично нульовий).

**Leakage.** `dropoff_at`, фактична тривалість, `trip_distance`, `fare_amount`, `tip_amount`, `total_amount` і `payment_type` стають відомими лише після завершення поїздки. Для прогнозу вартості чи тривалості в момент посадки вони не можуть бути ознаками, а `total_amount` при прогнозі `tip_amount` прямо містить відповідь.

**Pipeline.** Я запропонувала б додати в джерело стабільний `trip_id`, явно позначати сторно та їхній зв'язок з оригінальною поїздкою, передавати час у UTC або з offset, постачати словник кодів разом із файлом і додати автоматичні DQ-перевірки (діапазони, `dropoff >= pickup`, частка NULL за постачальником).

**ML-задача.** Природно підходять **регресія** (вартість, тривалість, чайові для карткових поїздок) і **forecasting** попиту за зонами й годинами; класифікацію можна будувати для задачі «чи залишить пасажир чайові».